### Dataset analysis for robot focus method
1. Find robot in image\
2. focus on the robot by "zooming in" cropping the surrounding.\

#### Get robot bboxes

In [ ]:
import os
import bz2
import pickle
import math
import numpy as np
from PIL import Image, ImageDraw
import cv2
from pathlib import Path
from tqdm.notebook import tqdm, trange
import matplotlib.pyplot as plt
import json
import h5py
import supervision as sv
from functools import wraps
from collections import defaultdict
from pathlib import Path

In [ ]:
def load_detection_from_h5(img_group) -> sv.Detections:
    xyxy = img_group["xyxy"][:]            # numpy array (N,4)
    masks = img_group["mask"][:]            # numpy array (N, H, W)
    confidence = img_group["confidence"][:] # numpy array (N,)
    class_id = img_group["class_id"][:]     # numpy array (N,)

    detection = sv.Detections(
        xyxy=xyxy,
        mask=masks,
        confidence=confidence,
        class_id=class_id,
    )
    return detection

def foreach_image_in_hdf5(confidence_threshold=0.3, deduplicate_bbox=True):
    def decorator(func):
        @wraps(func)
        def wrapper(hdf5_path, *args, **kwargs):
            with h5py.File(hdf5_path, "r") as f:
                results = []
                for category in tqdm(['Home', 'BigOffice-2', 'BigOffice-3', 'Hallway', 'MeetingRoom', 'SmallOffice'], desc='Category'):
                    for img_name in tqdm(f[category].keys(), desc='Image', leave=False):
                        img_group = f[category][img_name]

                        if deduplicate_bbox:
                            det = load_detection_from_h5(img_group).with_nms(class_agnostic=True)
                            conf, class_id, xyxy = det.confidence, det.class_id, det.xyxy
                        else:
                            conf = img_group["confidence"][...]
                            class_id = img_group["class_id"][...]
                            xyxy = img_group["xyxy"][...]

                        keep_indices = np.where(conf >= confidence_threshold)[0]

                        detections = {
                                'class_id': class_id[keep_indices],
                                'xyxy': xyxy[keep_indices]
                            }
                        
                        results.append(func(category, img_name, detections, *args, **kwargs))
            return results
        return wrapper
    return decorator

In [ ]:
# Deduplication takes longer but helps to get more valid samples where only 1 robot is detected.

@foreach_image_in_hdf5(confidence_threshold=0.3, deduplicate_bbox=True)
def robot_count_per_category_per_image(category, img_name, detections):
    robot_count = np.count_nonzero(detections['class_id'] == 1)
    return category, robot_count

results = robot_count_per_category_per_image('../data/autodistill_segmentation_dataset.hdf5')
robot_counts_per_cat = defaultdict(list)
for category, count in results:
    robot_counts_per_cat[category].append(count)

In [ ]:
all_counts = [c for counts in robot_counts_per_cat.values() for c in counts]
unique_counts, image_counts = np.unique(all_counts, return_counts=True)

for c, n in zip(unique_counts, image_counts):
    print(f"Robots: {c} | Images: {n}")

# Robots: 0 | Images: 33
# Robots: 1 | Images: 1193
# Robots: 2 | Images: 549
# Robots: 3 | Images: 167
# Robots: 4 | Images: 49
# Robots: 5 | Images: 8
# Robots: 6 | Images: 1

# if deduplicated:
# Robots: 0 | Images: 92
# Robots: 1 | Images: 1438
# Robots: 2 | Images: 408
# Robots: 3 | Images: 56
# Robots: 4 | Images: 5
# Robots: 5 | Images: 1

In [ ]:
for category, counts in robot_counts_per_cat.items():
    unique_counts, image_counts = np.unique(counts, return_counts=True)
    
    print(f"\nCategory {category}:")
    for c, n in zip(unique_counts, image_counts):
        print(f"Robots: {c} | Images: {n}")

# Category BigOffice-2:
# Robots: 0 | Images: 11
# Robots: 1 | Images: 187
# Robots: 2 | Images: 2

# Category BigOffice-3:
# Robots: 0 | Images: 4
# Robots: 1 | Images: 184
# Robots: 2 | Images: 12

# Category Hallway:
# Robots: 0 | Images: 10
# Robots: 1 | Images: 178
# Robots: 2 | Images: 12

# Category Home:
# Robots: 0 | Images: 39
# Robots: 1 | Images: 608
# Robots: 2 | Images: 292
# Robots: 3 | Images: 55
# Robots: 4 | Images: 5
# Robots: 5 | Images: 1

# Category MeetingRoom:
# Robots: 0 | Images: 28
# Robots: 1 | Images: 159
# Robots: 2 | Images: 13

# Category SmallOffice:
# Robots: 1 | Images: 122
# Robots: 2 | Images: 77
# Robots: 3 | Images: 1

In [ ]:
@foreach_image_in_hdf5(confidence_threshold=0.3, deduplicate_bbox=False)
def robot_bbox_per_image(category, img_name, detections):
    image_root = Path("../data/images")
    robot_indices  = np.where(detections['class_id'] == 1)[0]
    if len(robot_indices) == 1: #images where there is only 1 robot
        single_robot_index  = robot_indices[0]
        return {
            'image_path': (image_root / img_name).as_posix(),
            'bbox': detections['xyxy'][single_robot_index].tolist(),
        }
    return None

results = robot_bbox_per_image('../data/autodistill_segmentation_dataset.hdf5')
robot_bboxes = [r for r in results if r is not None]


In [ ]:
# with open(f"../data/robot_bboxes.json", "w") as f:
#     json.dump(robot_bboxes, f)

In [ ]:
with open("../data/robot_bboxes_deduplicated.json", "r") as f:
    robot_bboxes = json.load(f)

In [ ]:
#Inspect faulty bounding boxes manually

# output_dir = Path("./bbox_inspection")
# output_dir.mkdir(exist_ok=True)
# target_size = (128,128)

# for entry in tqdm(robot_bboxes):
#     img_path = entry['image_path']
#     bbox = entry['bbox']

#     img = Image.open(img_path).convert("RGB")
#     iw, ih = img.size

#     crop_box = get_crop_window(iw, ih, bbox, absolute_crop_size=None)

#     cropped_img = img.crop(crop_box)
#     resized_img = cropped_img.resize(target_size, Image.BILINEAR)

#     output_path = output_dir / Path(img_path).name
#     resized_img.save(output_path)

In [ ]:
# Exclude images that incorrectly labeled the robot - find them manually

exclude = [
    "Hallway_205", 
    "Hallway_273", 
    "Hallway_38",
    "38_1_0_4_3_0.7022904_0_1_1_2.686175_0.7022905_0.7022906_0.7022906_220_340_100_0_0_1_0_1_1_1_0_50_0_0_5_Pepper_TV",
    "4_1_0_6_4_0.5346667_0_1_1_2.360057_0.5346668_0.5346668_0.5346668_300_30_120_0_0_1_0_1_1_1_0_50_0_0_7_Pepper_TV",
    "576_1_0_3_5_0.6744272_0_1_1_1.406374_0.674427_0.674427_0.6744272_150_222_6_0_0_0_1_1_1_0_0_50_0_1_7_Pepper_TV",
    "SmallOffice_236",
    "62_1_0_4_3_0.59108_0_1_1_1.457013_0.59108_0.59108_0.59108_354_114_234_0_1_0_0_1_1_1_0_50_1_0_6_Pepper_TV",
    "440_1_0_5_3_0.7460655_0_1_1_2.220734_0.7460653_0.7460654_0.7460655_158_38.00001_278_0_0_1_0_1_0_0_1_1.334754_0_0_6_Pepper_TV",
    "161_0_1_3_0_50_50_1_0_0_1.987591_3.786283_5.79548_105.7212_142.4231_187.709_246.0405_0_0_0_0_1_0_0_50_0_0_4_Pepper_TV",
    "131_0_1_5_3_0.5728871_0_1_1_0_0.5728869_0.5728872_0.5728872_311_71_191_0_0_0_0_1_1_1_0_50_0_1_6_Pepper_TV",
    "595_1_0_6_0_50_50_0_0_1.915086_0.5006611_1.678949_2.274836_87.06186_181.4299_128.5675_340.4438_0_0_0_1_0_0_0_50_0_0_7_Pepper_TV",
    "601_0_1_6_0_50_50_1_0_0_0.5233274_1.153666_1.308424_270.5699_114.6801_182.1744_169.0011_0_0_0_0_0_0_0_50_0_0_7_Pepper_TV",
    "Hallway_44",
    "Hallway_76",
    "MeetingRoom_47",
    "MeetingRoom_69",
    "Hallway_269",
    "MeetingRoom_0",
    "SmallOffice_15",
    "SmallOffice_278",
    ]
robot_bboxes = [
    e for e in robot_bboxes
    if not any(k in e["image_path"] for k in exclude)
]

# with open(f"../data/robot_bboxes_deduplicated.json", "w") as f:
#     json.dump(robot_bboxes, f)

#### Prepare cropped robot closeups

In [2]:
from pathlib import Path
import numpy as np
import json
from PIL import Image, ImageDraw
from tqdm import tqdm
import os

In [ ]:
with open("../data/robot_bboxes_deduplicated.json", "r") as f:
    robot_bboxes = json.load(f)

In [ ]:
def get_max_crop_window(robot_bboxes):
    max_width = 0
    max_height = 0
    max_bbox_img=''

    for entry in robot_bboxes:
        bbox_img = entry['image_path']
        bbox = entry['bbox']
        w = bbox[2] - bbox[0]
        h = bbox[3] - bbox[1]

        if w > max_width:
            max_width = w
            max_bbox_img = bbox_img
        if h > max_height:
            max_height = h
            max_bbox_img = bbox_img
    return (round(max_width), round(max_height)), max_bbox_img


# Find 16:9 crop window enclosing the biggest bbox without cropping it (expand bbox to 16:9)
def expand_bbox_to_aspect(bbox_w, bbox_h, target_aspect=16/9):
    # Start from bbox size
    w, h = bbox_w, bbox_h
    current_aspect = w / h

    if current_aspect > target_aspect:
        # Too wide: increase height
        new_h = w / target_aspect
        new_w = w
    else:
        # Too tall: increase width
        new_w = h * target_aspect
        new_h = h
    return int(np.ceil(new_w)), int(np.ceil(new_h))

max_size, img_path = get_max_crop_window(robot_bboxes)
print(f"Biggest bbox: {max_size} {img_path}")
max_crop_width, max_crop_height = expand_bbox_to_aspect(*max_size)
print(f"Absolute crop window size (closest 16:9 enclosing biggest bbox): {max_crop_width}x{max_crop_height}")


In [ ]:
def get_crop_window(image_width, image_height, bbox, bbox_crop_ratio, absolute_crop_size=None):
    """
    Returns a crop window (x1, y1, x2, y2).
    - If absolute_crop_size is given (width, height), center crop window of that size on bbox center, clamped inside image.
    - Otherwise, calculate relative crop window relative to the bounding box with a bbox_crop_ratio. 
    bbox_crop_ratio=2 meaning that the crop will be twice the size of the bounding box

    bbox = [x1, y1, x2, y2]
    """
    x1, y1, x2, y2 = bbox
    bbox_w, bbox_h = x2 - x1, y2 - y1
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2

    if absolute_crop_size is not None:
        crop_w, crop_h = absolute_crop_size
    else:
        crop_w = crop_h = max(bbox_w, bbox_h) * bbox_crop_ratio 

    # Crop box coords
    left = int(round(cx - crop_w / 2))
    upper = int(round(cy - crop_h / 2))
    right = left + crop_w
    lower = upper + crop_h

    # Clamp inside image bounds
    if left < 0:
        right -= left  # Move right boundary
        left = 0
    if upper < 0:
        lower -= upper
        upper = 0
    if right > image_width:
        left -= (right - image_width)
        right = image_width
        if left < 0:
            left = 0
    if lower > image_height:
        upper -= (lower - image_height)
        lower = image_height
        if upper < 0:
            upper = 0

    return left, upper, right, lower


In [ ]:
# Create a dir of cropped robot closeups for training
for r in [1.5, 2, 3, 4]:
    target_size = (512,512)
    bbox_crop_ratio = r
    output_dir = Path(f"../data/robot_closeup_x{bbox_crop_ratio}")
    output_dir.mkdir(exist_ok=True)

    for entry in tqdm(robot_bboxes):
        img_path = entry['image_path']
        bbox = entry['bbox']

        img = Image.open(img_path).convert("RGB")
        iw, ih = img.size

        crop_box = get_crop_window(iw, ih, bbox, bbox_crop_ratio=bbox_crop_ratio, absolute_crop_size=None)

        cropped_img = img.crop(crop_box)
        resized_img = cropped_img.resize(target_size, Image.BILINEAR)

        output_path = output_dir / Path(img_path).name
        resized_img.save(output_path)


In [ ]:
# Create a dir of resized images for training

# target_size = (256,144)
# src_dir = Path('../data/images')
# dst_dir = Path(f"../data/images_{target_size[0]}x{target_size[1]}")
# dst_dir.mkdir(exist_ok=True)

# for path in tqdm(src_dir.glob('*.png')):
#     img = Image.open(path).convert("RGB")
#     resized_img = img.resize(target_size, Image.BILINEAR)
#     resized_img.save(dst_dir / path.name)


#### Prepare datasets for training

In [3]:
from data_utils import stratified_train_test_split, equalize_test_data
import pandas as pd

df = pd.read_pickle("../data/pepper_data.pkl")
images_focus = os.listdir('../data/robot_closeup_x2')

df = df[df['image_path'].apply(lambda x: Path(x).name).isin(images_focus)]
df['image_path'] = df['image_path'].apply(lambda p: (Path('../data/images') / Path(p).name).resolve().as_posix())

df.to_pickle("../data/robotfocus_pepper_data.pkl")

# df['image_path_scene'] = df['image_path'].apply(lambda p: (Path('../data/resized_images') / Path(p).name).as_posix())
# df['image_path_focus'] = df['image_path'].apply(lambda p: (Path('../data/robot_closeup') / Path(p).name).as_posix())


In [4]:
train_path, test_path = stratified_train_test_split(dataframe_path='../data/robotfocus_pepper_data.pkl', stratify_by_col='domain', seed=42)
test_equal_path = equalize_test_data(test_dataframe_path=test_path, seed=42)